# Content-Yield -> Full-Range Pack (headless GPU) -- v7

Builds the **authoritative full-range flop content pack** on a Kaggle **T4 GPU**, with **per-board checkpointing** so a single crashing board never aborts the run.

**Run (headless -- survives a closed browser):**
1. Settings -> **Accelerator -> GPU T4 x1**, **Internet -> On** (one-time phone verification may be required for both).
2. **Save Version -> Save & Run All (Commit)**. Close the tab; it runs in the background.
3. When done: version -> **Output** tab -> download `flop_pack_v1_fullrange.db` (+ `.db.gz`, `records_v1_fullrange.json`).

**What the checkpointing does and doesn't do.** Each board is solved, validated (NaN/inf records dropped, never signed into the pack), then written to `cyout/boards/board_XX.json`; `records.json` + `yield_report.json` refresh after every board. A single crashing board is logged and **skipped**, and the run continues. Note: a Batch commit that **exceeds Kaggle's ~9 h GPU limit** may be killed *without* saving `/kaggle/working`, so keep the run inside the limit -- the main run is ~5-7 h, and the smoke + optional raise cells auto-skip in a commit. Checkpoints let an **interactive** re-run resume; a fresh commit starts over (`/kaggle/working` is wiped between commits).

In [ ]:
# Fail fast if no GPU (before cloning the source) so a mis-set accelerator
# never silently falls back to a multi-day CPU run.
import subprocess
try:
    _r = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
except FileNotFoundError:
    raise SystemExit('No nvidia-smi (CPU instance) -- set Settings -> Accelerator -> GPU T4 x1, then re-run.')
assert _r.returncode == 0 and 'GPU' in _r.stdout, \
    'NO GPU detected -- set Settings -> Accelerator -> GPU T4 x1, then re-run.'
print(_r.stdout.strip())

In [ ]:
# Pull the solver source from the public repo (needs Internet On). Replaces the
# old embedded-zip cell, which was too large for Kaggle's kernel to run reliably.
!rm -rf /kaggle/working/poker && git clone -q --depth 1 https://github.com/tian-chaiyaporn2/poker_offline_trainer /kaggle/working/poker
import os
print('source ready ->', sorted(os.listdir('/kaggle/working/poker/src/pokertrainer'))[:6], '...')

In [ ]:
try:
    import cupy as cp
except Exception:
    import subprocess,sys; subprocess.run([sys.executable,'-m','pip','install','-q','cupy-cuda12x']); import cupy as cp
free,total=cp.cuda.Device(0).mem_info
nm=cp.cuda.runtime.getDeviceProperties(0)['name']; print('cupy',cp.__version__,'|',nm.decode() if isinstance(nm,bytes) else nm)
print('GPU memory: %.1f GB free / %.1f GB total'%(free/1e9,total/1e9))

### Smoke (optional, ~20–35 min) — 1 board at full range, float32. Confirms the GPU path before the long run.

In [ ]:
# Smoke pre-flight (~20-35 min): 1 board at full range, float32. Runs in an
# INTERACTIVE session to confirm the GPU path; AUTO-SKIPS in a Batch commit
# (Save & Run All) so it never delays or stops the full run below. All parsing is
# guarded so a smoke hiccup can never abort the commit.
import os, json
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'Interactive') == 'Batch':
    print('Batch commit -> skipping smoke (the full run below already covers board 0).')
else:
    import subprocess
    subprocess.run('cd /kaggle/working/poker && PYTHONPATH=src python -m pokertrainer.content_yield '
                   '--solver gpu --dtype float32 --n 400 --iters 300 --roots 0 --out /kaggle/working/cy_smoke',
                   shell=True)
    try:
        import cupy as cp
        print('peak GPU mem used: %.1f GB' % (cp.get_default_memory_pool().used_bytes() / 1e9))
        r = json.load(open('/kaggle/working/cy_smoke/yield_report.json'))
        print('config:', r.get('config'))
        print('boards completed:', r.get('boards_completed'),
              '| raw accepted/root:', r.get('projected_raw_accepted_per_root_full_range'))
    except Exception as e:
        print('smoke post-parse note (non-fatal):', e)

## Full run — 12 boards, full range, float32 (checkpointed, ~5–7 h)

In [ ]:
!cd /kaggle/working/poker && PYTHONPATH=src python -m pokertrainer.content_yield \
    --solver gpu --dtype float32 --n 400 --iters 300 --out /kaggle/working/cyout

In [ ]:
# Build + sign + verify the pack from whatever boards completed (partial-safe).
import subprocess, shutil, os, json, sys
env = {**os.environ, 'PYTHONPATH': 'src'}
try:
    rep = json.load(open('/kaggle/working/cyout/yield_report.json'))
except (FileNotFoundError, json.JSONDecodeError):
    raise SystemExit('No usable cyout/yield_report.json -- the full run produced no output. '
                     'Check the cell above and /kaggle/working/cyout/boards/*.ERROR.txt')
done, missing = rep.get('boards_completed') or [], rep.get('boards_missing') or []
print('boards completed:', done, '| missing:', missing)
# Refuse to build/sign an EMPTY pack: an all-boards-crash run must fail loudly,
# not hand back a signed 0-row pack that trivially "verifies".
if not done:
    raise SystemExit('NO boards completed -- refusing to build an empty signed pack. '
                     'Inspect /kaggle/working/cyout/boards/*.ERROR.txt to see why every board failed.')
if missing:
    print(f'*** NOTE: {len(missing)} board(s) missing {missing} -- pack is PARTIAL (still usable). ***')
subprocess.run(['python', '-m', 'pokertrainer.content_pack',
                '--records', '/kaggle/working/cyout/records.json',
                '--version', 'v1_fullrange', '--out', '/kaggle/working'],
               cwd='/kaggle/working/poker', env=env, check=True)
shutil.copy('/kaggle/working/cyout/records.json', '/kaggle/working/records_v1_fullrange.json')
sys.path.insert(0, '/kaggle/working/poker/src')
from pokertrainer.content_pack import verify_pack
db = '/kaggle/working/flop_pack_v1_fullrange.db'
v = verify_pack(db)
print('VERIFY:', v, '| size MB:', round(os.path.getsize(db) / 1e6, 2))
assert v['hash_ok'] and v['signature_ok'], f'PACK INTEGRITY FAILED: {v}'
assert v['records'] > 0, f'EMPTY pack: {v}'
print(json.dumps({k: rep[k] for k in ('accepted', 'accepted_deduped', 'distinct_concepts_measured',
                                      'per_node_accepted', 'coverage') if k in rep}, indent=1))

---
### Optional -- raise-inclusive full range (FR-011)
fold/call/**raise** ~triples solve time, so all 12 boards won't fit one ~9 h commit.
**Off by default** (`RUN_RAISE_HALF = False`) so Save & Run All stops after the main pack.
To run: set `RUN_RAISE_HALF = True`, pick `HALF='A'` (boards 0-5) or `'B'` (6-11), Save & Run All (one commit each).
Download both `records_raise_*.json`; merge + sign the raise pack locally afterwards.

In [ ]:
# Gated off by default so Save & Run All does not start a multi-hour raise solve
# after the main pack (that combination exceeds a typical 12 h Kaggle commit).
# Set RUN_RAISE_HALF = True only when this commit is intentionally for FR-011.
RUN_RAISE_HALF = False
HALF = 'A'   # <-- 'A' for one commit, 'B' for the other
if not RUN_RAISE_HALF:
    print('Skipping optional raise half (set RUN_RAISE_HALF=True to enable).')
else:
    ROOTS = {'A': '0,1,2,3,4,5', 'B': '6,7,8,9,10,11'}[HALF]
    import os, shutil, subprocess
    env = {**os.environ, 'PYTHONPATH': 'src'}
    subprocess.run(
        ['python', '-m', 'pokertrainer.content_yield',
         '--solver', 'gpu', '--dtype', 'float32', '--n', '400', '--iters', '300',
         '--raise-x', '3', '--roots', ROOTS,
         '--out', f'/kaggle/working/cy_raise_{HALF}'],
        cwd='/kaggle/working/poker', env=env, check=True,
    )
    shutil.copy(f'/kaggle/working/cy_raise_{HALF}/records.json',
                f'/kaggle/working/records_raise_{HALF}.json')
    print('done half', HALF)